# Open Science Replication Notebook: Word Co-occurrence Networks and Multi-Dimensional Sociodemographic Stratification in the 1988 Brazilian Constituent Assembly (SAIC Dataset)

This notebook reproduces the complete data curation, Natural Language Processing (NLP), Positive Pointwise Mutual Information (PPMI) semantic network modeling, Louvain community detection, and multi-dimensional sociodemographic, geographic, and political analyses of the **72,719 citizen letters** submitted to the 1988 Brazilian National Constituent Assembly via the **SAIC (*Sistema de Apoio Informático ao Cidadão*)** historical database.

## 1. Environment Setup and Package Imports

In [ ]:
# Uncomment the line below when running in Google Colab:
# !pip install -q spacy scikit-learn networkx matplotlib scipy pandas numpy && python -m spacy download pt_core_news_md

import os
import urllib.request
import numpy as np
import pandas as pd
import spacy
import networkx as nx
from scipy.stats import chi2_contingency
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.makedirs('figures', exist_ok=True)

## 2. Automated Data Acquisition & Curation Audit (`SAIC.csv`)

If `SAIC.csv` is not present locally, it is automatically downloaded from the companion Open Science repository (`https://github.com/alkawamo/base-saic`).

In [ ]:
CSV_PATH = 'SAIC.csv'
if not os.path.exists(CSV_PATH):
    print('Downloading curated SAIC.csv from GitHub (alkawamo/base-saic)...')
    url = 'https://raw.githubusercontent.com/alkawamo/base-saic/main/SAIC.csv'
    urllib.request.urlretrieve(url, CSV_PATH)

df = pd.read_csv(CSV_PATH, sep=';', low_memory=False)
print(f'Dataset dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns')

audit_rows = []
for col in df.columns:
    audit_rows.append({
        'Column (PT)': col,
        'Missing (NaN)': int(df[col].isna().sum()),
        'Completeness (%)': round((1 - df[col].isna().mean()) * 100, 2),
        'Unique Values': int(df[col].nunique(dropna=True))
    })
display(pd.DataFrame(audit_rows))

## 3. NLP Preprocessing & Lemmatization (`spaCy pt_core_news_md`)

To prevent organized petition campaigns with identical boilerplate text from artificially inflating word co-occurrence probabilities, we deduplicate `TEXTO SUGESTÃO` during the construction of the PPMI network ($N = 70,151$ unique valid texts), and then project all $72,719$ records onto the resulting community space for sociodemographic and geographic stratification.

In [ ]:
df_dedup = df.dropna(subset=['TEXTO SUGESTÃO']).drop_duplicates(subset=['TEXTO SUGESTÃO']).copy()
print(f'Unique valid citizen letters (N_unique): {len(df_dedup):,}')

nlp = spacy.load('pt_core_news_md', disable=['ner', 'parser'])
texts_dedup = df_dedup['TEXTO SUGESTÃO'].astype(str).tolist()

docs_dedup = []
for doc in nlp.pipe(texts_dedup, batch_size=1000):
    lemmas = [
        t.lemma_.lower().strip() for t in doc
        if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha and len(t.lemma_.strip()) > 2
    ]
    docs_dedup.append(' '.join(lemmas))

text_to_lemma = dict(zip(texts_dedup, docs_dedup))
all_lemmas = [text_to_lemma.get(str(t) if pd.notna(t) else '', '') for t in df['TEXTO SUGESTÃO']]
print('Lemmatization complete!')

## 4. PPMI Word Co-occurrence Network & Parameter Sensitivity Analysis

We evaluate document frequency thresholds (`min_df` $\in \{1\%, 1.5\%, 2\%, 2.5\%, 3\%\}$) and PPMI percentile cutoffs ($\{50, 75, 85, 90, 95\}$), selecting `min_df = 2.0%` and the **95th percentile PPMI cutoff ($\theta_{95} = 1.25$)**, which maximizes modularity ($Q = 0.6363$) while preserving a single connected component ($|V| = 322, |E| = 1,823$).

In [ ]:
vec = CountVectorizer(binary=True, min_df=0.02)
A_dedup = vec.fit_transform(docs_dedup)
vocab = np.array(vec.get_feature_names_out())
n_docs, n_terms = A_dedup.shape

C = (A_dedup.T @ A_dedup).toarray().astype(float)
diag = np.diag(C).copy()
np.fill_diagonal(C, 0.0)

p_ij = C / n_docs
p_i = diag / n_docs
with np.errstate(divide='ignore', invalid='ignore'):
    pmi = np.log2(p_ij / np.outer(p_i, p_i))
ppmi = np.maximum(0.0, np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=0.0))

iu = np.triu_indices(n_terms, k=1)
pos_vals = ppmi[iu][ppmi[iu] > 0]
corte_95 = np.percentile(pos_vals, 95)

G = nx.Graph()
mask = ppmi[iu] >= corte_95
for u_idx, v_idx, w in zip(iu[0][mask], iu[1][mask], ppmi[iu][mask]):
    G.add_edge(vocab[u_idx], vocab[v_idx], weight=float(w))

comms = nx.community.louvain_communities(G, weight='weight', seed=42)
Q_opt = nx.community.modularity(G, comms, weight='weight')
print(f'Optimal Network: |V|={G.number_of_nodes()}, |E|={G.number_of_edges()}, Density={nx.density(G)*100:.2f}%, Q={Q_opt:.4f}')

## 5. Community Characterization, Centralities & Figure 1 (Co-occurrence Graph)

In [ ]:
comm_defs = [
    ('C1: Economic Dev. & Agriculture', {'agricultura', 'agrícola', 'terra', 'produzir', 'inflação'}, '#2ca02c'),
    ('C2: Public Admin. & State Org.', {'união', 'cargo', 'concurso', 'funcionário', 'ministério'}, '#1f77b4'),
    ('C3: Political & Electoral Structure', {'congresso', 'deputado', 'senador', 'mandato', 'eleição'}, '#ff7f0e'),
    ('C4: Educational Policy', {'ensino', 'escola', 'universidade', 'educação', 'gratuito'}, '#d62728'),
    ('C5: Penal System & Public Security', {'prisão', 'crime', 'polícia', 'violência', 'aborto', 'pena'}, '#9467bd'),
    ('C6: Social Security & Healthcare', {'previdência', 'inps', 'aposentadoria', 'aposentado', 'hospital'}, '#8c564b'),
    ('C7: Professional Autonomy & Labor', {'detetive', 'investigação', 'autonomia', 'autônomo', 'liberal'}, '#e377c2'),
    ('C8: Popular Appeals & Protests', {'deus', 'gente', 'fome', 'sarney', 'pobre', 'sofrer'}, '#17becf')
]

short_names = [
    'C1: Econ/Agri', 'C2: PubAdmin', 'C3: Politics', 'C4: Education',
    'C5: Penal/Sec', 'C6: SocSec/Health', 'C7: ProfAuton', 'C8: PopAppeals'
]

comm_to_nodes, node_to_comm = {}, {}
for c in comms:
    best_lbl = max(comm_defs, key=lambda x: len(c.intersection(x[1])))[0]
    comm_to_nodes[best_lbl] = c
    for n in c:
        node_to_comm[n] = best_lbl

ordered_labels = [x[0] for x in comm_defs]
colors = [x[2] for x in comm_defs]

# Project all 72,719 letters into the 8 communities (size-normalized affinity)
A_all = vec.transform(all_lemmas)
vocab_to_idx = {w: i for i, w in enumerate(vocab)}
M_tc = np.zeros((n_terms, len(ordered_labels)), dtype=float)
comm_sizes = np.zeros(len(ordered_labels), dtype=float)
for j, lbl in enumerate(ordered_labels):
    comm_sizes[j] = len(comm_to_nodes[lbl])
    for w in comm_to_nodes[lbl]:
        if w in vocab_to_idx:
            M_tc[vocab_to_idx[w], j] = 1.0

D_all_raw = A_all @ M_tc
D_all_norm = D_all_raw / comm_sizes[None, :]
row_sums = D_all_norm.sum(axis=1, keepdims=True)
D_all_prop = np.divide(D_all_norm, row_sums, where=row_sums > 0)
has_terms = (D_all_raw.sum(axis=1) > 0)
df['DOM_COMM'] = np.where(has_terms, D_all_norm.argmax(axis=1), -1)

# Plot Figure 1
pos = {}
angles = np.linspace(0, 2 * np.pi, len(ordered_labels), endpoint=False)
for idx, lbl in enumerate(ordered_labels):
    subG = G.subgraph(comm_to_nodes[lbl])
    sub_pos = nx.spring_layout(subG, weight='weight', k=0.45, iterations=100, seed=42)
    cx, cy = 3.2 * np.cos(angles[idx]), 3.2 * np.sin(angles[idx])
    for n, (x, y) in sub_pos.items():
        pos[n] = np.array([cx + x * 0.95, cy + y * 0.95])

fig, ax = plt.subplots(figsize=(11, 8), dpi=300)
inter_edges = [(u, v) for u, v in G.edges() if node_to_comm[u] != node_to_comm[v]]
intra_edges = [(u, v) for u, v in G.edges() if node_to_comm[u] == node_to_comm[v]]
nx.draw_networkx_edges(G, pos, edgelist=inter_edges, alpha=0.08, edge_color='#888888', width=0.5, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=intra_edges, alpha=0.35, edge_color='#444444', width=0.9, ax=ax)
bet = nx.betweenness_centrality(G)
patches = []
for idx, lbl in enumerate(ordered_labels):
    nodelist = list(comm_to_nodes[lbl])
    sizes = [60 + 600 * bet[n] for n in nodelist]
    nx.draw_networkx_nodes(G, pos, nodelist=nodelist, node_color=colors[idx], node_size=sizes, alpha=0.9, edgecolors='white', linewidths=0.5, ax=ax)
    patches.append(mpatches.Patch(color=colors[idx], label=f'{lbl} ({len(nodelist)} nodes)'))
    for n, _ in sorted(G.subgraph(nodelist).degree(), key=lambda x: x[1], reverse=True)[:3]:
        x, y = pos[n]
        ax.text(x, y + 0.08, n, fontsize=7.5, fontweight='bold', ha='center', va='bottom',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.75, edgecolor='none'))
ax.legend(handles=patches, loc='lower center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=True, fontsize=8.5)
ax.set_title('1988 Brazilian Constituent Assembly Word Co-occurrence Network (min_df = 2%, 95th Perc. PPMI, Q = 0.6363)', fontsize=11, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('figures/fig1_cooccurrence_network_en.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Top-Down (1986 Prodasen Catalog) vs. Bottom-Up (Louvain Communities) Alignment (Figure 2)

In [ ]:
theme_en_map = {
    'Direito - Organização do Estado': 'Law - State Organization',
    'Organização Social': 'Social Organization',
    'Política Educacional': 'Educational Policy',
    'Política Trabalhista': 'Labor Policy',
    'Política - Estrutura do Estado': 'Politics - State Structure',
    'Política Agrícola': 'Agricultural Policy',
    'Política Econômica': 'Economic Policy',
    'Administração de Pessoal - Serviço Público': 'Personnel Admin. - Civil Service',
    'Política Previdenciária': 'Social Security Policy',
    'Política Financeira': 'Financial Policy',
    'Política Nacional de Saúde': 'National Health Policy',
    'Sugestões Não Pertinentes': 'Non-Pertinent Suggestions',
    'Meio Ambiente - Ecologia': 'Environment & Ecology',
    'Política Nacional de Transportes': 'National Transport Policy',
    'Política Habitacional': 'Housing Policy',
    'Manifestação': 'Public Manifestation / Protest',
    'Telecomunicações': 'Telecommunications',
    'Política Industrial': 'Industrial Policy',
    'Política Energética': 'Energy Policy',
    'Política Indigenista': 'Indigenous Policy',
    'Política Nacional de Urbanismo': 'National Urban Policy',
    'Pedido de Informação sobre a Constituinte': 'Constituent Information Request',
    'Política Sanitária': 'Sanitation Policy',
    'Política Nacional da Informática': 'National Informatics Policy'
}
ordered_themes_pt = list(theme_en_map.keys())
cont_matrix = np.zeros((len(ordered_themes_pt), len(ordered_labels)), dtype=float)
for i, cat_str in enumerate(df['CATÁLOGO']):
    if pd.isna(cat_str) or not has_terms[i]:
        continue
    for t in [x.strip() for x in str(cat_str).split(',')]:
        if t in theme_en_map:
            cont_matrix[ordered_themes_pt.index(t), :] += D_all_prop[i, :]
cont_pct = cont_matrix / cont_matrix.sum(axis=1, keepdims=True) * 100.0

fig, ax = plt.subplots(figsize=(11, 8.5), dpi=300)
im = ax.imshow(cont_pct, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(short_names)))
ax.set_xticklabels(short_names, rotation=30, ha='right', fontsize=9, fontweight='bold')
ax.set_yticks(range(len(ordered_themes_pt)))
ax.set_yticklabels([theme_en_map[k] for k in ordered_themes_pt], fontsize=8.5)
for i in range(len(ordered_themes_pt)):
    for j in range(len(short_names)):
        val = cont_pct[i, j]
        ax.text(j, i, f'{val:.1f}%', ha='center', va='center', color='white' if val > 24 else 'black', fontsize=7.5)
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='Normalized Thematic Affinity (%)')
ax.set_title('Cross-Tabulation: 1986 Official Catalog (Top-Down) vs. Network Communities (Bottom-Up)', fontsize=11, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('figures/fig2_topdown_vs_bottomup_heatmap_en.png', dpi=300)
plt.show()

## 7. Multi-Dimensional Sociodemographic Stratification: Gender, Urban/Rural, Education, and Income (Figure 3)

In [ ]:
def get_group_distribution(col_name, value_map_ordered):
    mat = np.zeros((len(value_map_ordered), len(ordered_labels)))
    counts = []
    keys_pt = list(value_map_ordered.keys())
    for g_idx, k_pt in enumerate(keys_pt):
        mask_g = (df[col_name] == k_pt) & has_terms
        counts.append(mask_g.sum())
        if mask_g.sum() > 0:
            mat[g_idx, :] = D_all_prop[mask_g].mean(axis=0) * 100.0
    valid_mask = df[col_name].isin(keys_pt) & (df['DOM_COMM'] >= 0)
    ct = pd.crosstab(df.loc[valid_mask, col_name], df.loc[valid_mask, 'DOM_COMM'])
    chi2, pval, _, _ = chi2_contingency(ct)
    n_tot = ct.values.sum()
    cramers_v = np.sqrt(chi2 / (n_tot * (min(ct.shape) - 1)))
    print(f"{col_name}: N={n_tot:,}, Chi2={chi2:.1f}, p={pval:.2e}, Cramer's V={cramers_v:.4f}")
    return mat, [f"{value_map_ordered[k]} (n={counts[i]:,})" for i, k in enumerate(keys_pt)], chi2, cramers_v

sex_map = {'Masculino': 'Male', 'Feminino': 'Female'}
res_map = {'Zona urbana': 'Urban Area', 'Zona rural': 'Rural Area'}
edu_map = {
    'Analfabeto(a)': 'Illiterate', 'Primeiro grau incompleto': 'Primary Incomp.',
    'Primeiro grau completo': 'Primary Comp.', 'Segundo grau incompleto': 'High School Incomp.',
    'Segundo grau completo': 'High School Comp.', 'Superior incompleto': 'Undergrad. Incomp.',
    'Superior completo': 'Undergrad. Comp.', 'Pós-graduação': 'Postgraduate'
}
inc_map = {
    'Sem rendimento': 'No Income', 'Até 1 salário mínimo': 'Up to 1 MW',
    'Mais de 01 a 02 s.m.': '1 to 2 MW', 'Mais de 02 a 03 s.m.': '2 to 3 MW',
    'Mais de 03 a 05 s.m.': '3 to 5 MW', 'Mais de 05 a 10 s.m.': '5 to 10 MW',
    'Mais de 10 a 20 s.m.': '10 to 20 MW', 'Mais de 20 s.m.': '> 20 MW'
}

mat_sex, lbl_sex, chi_sex, v_sex = get_group_distribution('SEXO', sex_map)
mat_res, lbl_res, chi_res, v_res = get_group_distribution('MORADOR', res_map)
mat_edu, lbl_edu, chi_edu, v_edu = get_group_distribution('INSTRUÇÃO', edu_map)
mat_inc, lbl_inc, chi_inc, v_inc = get_group_distribution('FAIXA RENDA', inc_map)

## 8. Macro-Regional Geography (Figure 4) & Citizen–Parliamentarian Bipartite Network (Figure 5)

In [ ]:
uf_to_region = {
    'AC': 'North', 'AM': 'North', 'AP': 'North', 'PA': 'North', 'RO': 'North', 'RR': 'North',
    'AL': 'Northeast', 'BA': 'Northeast', 'CE': 'Northeast', 'MA': 'Northeast', 'PB': 'Northeast',
    'PE': 'Northeast', 'PI': 'Northeast', 'RN': 'Northeast', 'SE': 'Northeast',
    'DF': 'Center-West', 'GO': 'Center-West', 'MS': 'Center-West', 'MT': 'Center-West',
    'ES': 'Southeast', 'MG': 'Southeast', 'RJ': 'Southeast', 'SP': 'Southeast',
    'PR': 'South', 'RS': 'South', 'SC': 'South'
}
df['REGION'] = df['UF'].map(uf_to_region)
reg_order = {k: k for k in ['North', 'Northeast', 'Center-West', 'Southeast', 'South']}
mat_reg, lbl_reg, chi_reg, v_reg = get_group_distribution('REGION', reg_order)

top_dest = df.loc[has_terms, 'DESTINATÁRIO'].dropna().value_counts().head(15)
for d_name, d_cnt in top_dest.items():
    mask_d = (df['DESTINATÁRIO'] == d_name) & has_terms
    vec_d = D_all_prop[mask_d].mean(axis=0) * 100.0
    print(f'{d_name:30s} (n={d_cnt:4d}): Dominant = {short_names[vec_d.argmax()]} ({vec_d.max():.1f}%)')